# 08 — Group GEMM

**Ladder:** Sequential `torch.matmul` → sequential tutorial-03 Triton matmul → (official) one-launch Group GEMM with a pointer table.

**Study tip.** Tutorial-03 **grouped L2 pid** ≠ **Group GEMM**. This notebook's Triton path is the sequential baseline; the pointer-table one-launch design is the next mental step.

Official: [Group GEMM](https://triton-lang.org/main/getting-started/tutorials/08-group-gemm.html)

Primer: [`../03_matrix_multiplication/README.md`](../03_matrix_multiplication/README.md) · figures: [`docs/images/`](docs/images/)

![Whiteboard: $G$ host launches vs one group-GEMM pointer table](docs/images/benchmark-group-gemm.png)

*Theory board — sequential launches vs device arrays of pointers + $(M_g,N_g,K_g)$ in one kernel.*

---

# Part I — Mathematics

## 0. Problem family and its algebra

A **group** is a list of independent products

$$
C^{(g)}=A^{(g)}B^{(g)},
\qquad
A^{(g)}\in\mathbb{R}^{M_g\times K_g},
\quad
B^{(g)}\in\mathbb{R}^{K_g\times N_g},
\quad
g=1,\dots,G.
\tag{0.1}
$$

The shapes may differ across $g$; only the inner dimensions must match within
each group.

**Proposition 1 (block-diagonal embedding).** Define

$$
\mathcal A=
\begin{pmatrix}
A^{(1)} & & \\
& \ddots & \\
& & A^{(G)}
\end{pmatrix},
\qquad
\mathcal B=
\begin{pmatrix}
B^{(1)} & & \\
& \ddots & \\
& & B^{(G)}
\end{pmatrix},
\tag{0.2}
$$

with $\mathcal A\in\mathbb{R}^{(\sum_gM_g)\times(\sum_gK_g)}$ and
$\mathcal B\in\mathbb{R}^{(\sum_gK_g)\times(\sum_gN_g)}$. Then

$$
\mathcal A\mathcal B=
\operatorname{diag}\!\left(C^{(1)},\dots,C^{(G)}\right).
\tag{0.3}
$$

*Proof.* Block multiplication: the $(g,h)$ block of $\mathcal A\mathcal B$ is
$A^{(g)}B^{(h)}$ if $g=h$ and $0$ otherwise, because the block-diagonal
structure forces the intermediate index to lie in block $g$. $\square$

So group GEMM is **one sparse (block-diagonal) GEMM**. A dense GEMM would
compute all $G^{2}$ cross-blocks and waste work; a group kernel enumerates only
the $G$ useful blocks.

**FLOPs.**

$$
F=\sum_{g=1}^{G}2M_gN_gK_g .
\tag{0.4}
$$

**Arithmetic intensity.** If each $A^{(g)},B^{(g)}$ is read once and
$C^{(g)}$ written once,

$$
Q=\sum_g\big(M_gK_g+K_gN_g+M_gN_g\big)b,
\qquad
\mathrm{AI}=\frac{F}{Q}.
\tag{0.5}
$$

For square groups $M_g=N_g=K_g=n$ this is
$\mathrm{AI}\approx2n^{3}/(3n^{2}b)=2n/(3b)$: small groups are memory-bound,
large groups compute-bound. That is the quantitative reason grouped execution
matters most for **many small** problems.

### 0.1 Launch-cost model

Let $L$ be the fixed cost of a kernel launch (host + driver + device setup) and
$t_g$ the GPU time of group $g$. Sequential execution on one stream takes

$$
T_{\mathrm{seq}}=\sum_{g=1}^{G}(L+t_g)=G\,L+\sum_g t_g .
\tag{0.6}
$$

A single-launch group kernel pays $L$ once:

$$
T_{\mathrm{group}}=L+\sum_g t_g+\varepsilon_{\mathrm{sched}},
\tag{0.7}
$$

where $\varepsilon_{\mathrm{sched}}$ is the cost of reading per-group metadata
and mapping tiles. Hence the speedup is

$$
\frac{T_{\mathrm{seq}}}{T_{\mathrm{group}}}
=
\frac{G\,L+\sum_g t_g}{L+\sum_g t_g+\varepsilon}
\;\xrightarrow[\;L\gg t_g\;]{}\;G .
\tag{0.8}
$$

When the $t_g$ dominate, the speedup tends to $1$: grouping is a **launch
amortization** technique, not an arithmetic one.

### 0.2 Tile enumeration and prefix sums

With tile sizes $B_M,B_N$, group $g$ has

$$
T_g=
\left\lceil\frac{M_g}{B_M}\right\rceil
\left\lceil\frac{N_g}{B_N}\right\rceil
\tag{0.9}
$$

tiles. The total is $T=\sum_gT_g$. Define the prefix sums

$$
P_0=0,
\qquad
P_g=\sum_{h=1}^{g}T_h,
\qquad g=1,\dots,G.
\tag{0.10}
$$

A global tile id $t\in[T]$ belongs to group

$$
g(t)=\min\{g:\ t<P_g\},
\tag{0.11}
$$

i.e. the group found by binary search in the prefix array. Within the group,

$$
t'=t-P_{g-1},
\qquad
p_m=\left\lfloor\frac{t'}{G_{N,g}}\right\rfloor,
\qquad
p_n=t'\bmod G_{N,g},
\tag{0.12}
$$

where $G_{N,g}=\lceil N_g/B_N\rceil$. **Proposition 2.** The map
$t\mapsto(g,p_m,p_n)$ is a bijection onto the disjoint union of all group tile
grids; hence every output tile is computed exactly once.

*Proof.* The intervals $[P_{g-1},P_g)$ partition $[T]$, and (0.12) is the
standard quotient–remainder bijection within each group. $\square$

This prefix-sum construction is the device-side "scheduler" of a one-launch
Group GEMM. It needs only the arrays $(T_g)$ or, equivalently,
$(M_g,N_g,B_M,B_N)$, which are $O(G)$ metadata.

### 0.3 Load balance

The work per tile is not uniform:

$$
\text{work}(g)\approx2B_MB_NDK_g
\tag{0.13}
$$

when the tile shape is fixed, because the reduction length is $K_g$. Groups with
large $K_g$ cost proportionally more. Two standard policies:

1. **Static.** Assign tile ids by prefix sum; simple, deterministic, but can
   leave a long group at the tail of the grid.
2. **Work-stealing.** A global atomic counter hands out the next tile id; idle
   programs take more work. This equalizes finish times at the cost of atomic
   traffic (the same trade-off as the $d\gamma$ reduction in notebook 05).

### 0.4 Contrast with `torch.bmm`

`torch.bmm` requires all items to share the same $(M,N,K)$. Padding every group
to the maximum shape gives

$$
F_{\mathrm{padded}}
=
2G\,\max_gM_g\,\max_gN_g\,\max_gK_g
\;\ge\;
F,
\tag{0.14}
$$

with equality only when all shapes coincide. For heterogeneous MoE experts the
waste can be an order of magnitude. Group GEMM avoids both the padding FLOPs and
the padded memory.

**Teaching shapes (below).** $(128\times64)(64\times256)$ and
$(64\times128)(128\times32)$:

$$
F=2(128)(256)(64)+2(64)(32)(128)
=4{,}194{,}304+524{,}288
=4{,}718{,}592\ \text{FLOPs}.
$$

With $B_M=B_N=64$: $T_1=2\cdot4=8$, $T_2=1\cdot1=1$, so $T=9$ tiles.

### Theory board

![Whiteboard: $G$ host launches vs one group-GEMM pointer table](docs/images/whiteboard-theory.jpg)

*Theory board — sequential launches vs device arrays of pointers + $(M_g,N_g,K_g)$ in one kernel.*

# Part II — Proof

**Theorem (block-diagonal embedding).** Define
$\mathcal A=\operatorname{diag}(A^{(1)},\dots,A^{(G)})$ and
$\mathcal B=\operatorname{diag}(B^{(1)},\dots,B^{(G)})$. Then
$\mathcal A\mathcal B=\operatorname{diag}(C^{(1)},\dots,C^{(G)})$.

*Proof.* Block multiplication: the $(g,h)$ block is $A^{(g)}B^{(h)}$ when
$g=h$ and zero otherwise. $\square$

**Theorem (prefix-sum tile map).** With $T_g$ tiles in group $g$ and prefix
sums $P_g=\sum_{h\le g}T_h$, the map
$t\mapsto(g,p_m,p_n)$ defined by $g=\min\{g:t<P_g\}$ and (0.12) is a bijection
onto the disjoint union of the group tile grids.

*Proof.* The intervals $[P_{g-1},P_g)$ partition $[T]$; within each group the
quotient–remainder map is a bijection. $\square$

**Corollary (one-launch correctness).** A single grid of $T$ programs, each
running the tile body for its assigned group and tile, computes all
$C^{(g)}$ correctly.

*Proof.* The bijection gives exactly-once coverage; each tile body is the
blocked product of notebook 03. $\square$

# Part III — Step-by-step proof

1. **Enumerate groups.** For each $g$, read $(M_g,N_g,K_g)$.
2. **Count tiles.** $T_g=\lceil M_g/B_M\rceil\lceil N_g/B_N\rceil$.
3. **Prefix sum.** Build $P_0=0,\dots,P_G=T$.
4. **Assign tile ids.** Program $t\in[T]$ finds its group by the prefix sum.
5. **Local tile.** $t'=t-P_{g-1}$, then
   $p_m=\lfloor t'/G_{N,g}\rfloor$ and
   $p_n=t'\bmod G_{N,g}$.
6. **Pointers.** Read the group's $A,B,C$ base offsets from the metadata.
7. **Tile body.** Run the $K$-panel loop and `tl.dot` of notebook 03.
8. **Store.** Write $C^{(g)}$ at the group's output offset.
9. **Launch once.** The host issues one kernel launch instead of $G$ launches;
   the arithmetic is unchanged.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import importlib.util

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")


try:
    _HERE = Path(__file__).resolve().parent
except NameError:  # Jupyter notebook: __file__ is undefined
    _HERE = Path.cwd()
_REPO = _HERE
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "03_matrix_multiplication" / "tutorial.py").exists():
        _REPO = _candidate
        break
else:
    raise FileNotFoundError(
        "cannot locate 03_matrix_multiplication from " + str(_HERE)
    )
_matmul_path = _REPO / "03_matrix_multiplication" / "tutorial.py"
_spec = importlib.util.spec_from_file_location("matmul_mod", _matmul_path)
_matmul_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_matmul_mod)
matmul = _matmul_mod.matmul


### Method 01 — Naive (sequential PyTorch)

For each $g$, independently,

$$
C^{(g)}_{mn}=\sum_kA^{(g)}_{mk}B^{(g)}_{kn}.
\tag{2.1}
$$

| Stage | How work is launched | Algebra |
|---|---|---|
| **Naive** | Python `for` + `torch.matmul` | Exact $C^{(g)}=A^{(g)}B^{(g)}$ |
| **Intermediate** | Same loop + tutorial-03 `matmul` | Same $C^{(g)}$; still $G$ grids |
| **Triton Group GEMM** | One grid; device pointer/size tables | Same $C^{(g)}$; amortized launch |

**Proposition 3 (independence).** Running `torch.matmul` or tutorial-03
`matmul` in a Python `for` loop yields the Cartesian product of the individual
solutions — there is no coupling between problems.

*Proof.* The maps $(A^{(g)},B^{(g)})\mapsto C^{(g)}$ act on disjoint data; the
composition of independent functions is the tuple of results. $\square$

### 2.1 Pointer table (official rung)

Host packs, on device,

$$
\texttt{A\_ptrs}[g],\quad
\texttt{B\_ptrs}[g],\quad
\texttt{C\_ptrs}[g],\quad
M[g],\quad N[g],\quad K[g].
\tag{2.2}
$$

A program reads its group's metadata and runs the familiar
$B_M\times B_N$ tile body (K-panels + `tl.dot`) from notebook 03. The only new
arithmetic is the prefix-sum lookup of §0.2.

**Pointer arithmetic.** For group $g$ and tile $(p_m,p_n)$,

$$
A_{\mathrm{ptrs}}
=
\texttt{A\_ptrs}[g]
+p_mB_Ms_{am}
+o_k s_{ak},
\qquad
B_{\mathrm{ptrs}}
=
\texttt{B\_ptrs}[g]
+o_ks_{bk}
+p_nB_Ns_{bn}.
\tag{2.3}
$$

The strides may differ per group (e.g. different leading dimensions), which is
why the table stores pointers rather than a single base plus a formula.

### 2.2 The sequential baseline in this notebook

`naive_group_gemm_pseudo` loops over `(A, B)` pairs with `torch.matmul`;
`group_gemm_sequential` uses the tutorial-03 Triton matmul for each pair. Both
are correct and both pay $G$ launches. They exist to make the launch-overhead
argument measurable: time them against a single-launch kernel on many small
problems and the $G\,L$ term becomes visible.

### 2.3 Why not concatenate and pad?

Padding to a common shape gives one dense GEMM, but it computes

$$
2G\,M_{\max}N_{\max}K_{\max}
\ \text{FLOPs}
\tag{2.4}
$$

instead of $\sum_g2M_gN_gK_g$. For $G=8$ experts with shapes varying by $4\times$,
the padding waste can exceed $50\%$. Group GEMM avoids it by enumerating exactly
the real tiles.

### 2.4 Load balancing in one launch

Even with a perfect tile map, the **duration** of tiles varies with $K_g$
(0.13). A static prefix-sum schedule processes groups in order, so the critical
path is the sum of the slowest tile in each wave. A dynamic atomic counter,

$$
t\leftarrow\operatorname{atomicAdd}(\mathrm{counter},1),
\tag{2.5}
$$

lets fast programs absorb the tail. The counter is a single integer, so
contention is negligible compared with the $O(T)$ work; the trade-off is loss of
determinism in the assignment (though each tile is still computed once).

### 2.5 Summary

| Concern | Sequential | One-launch group |
|---|---|---|
| Launches | $G$ | $1$ |
| Metadata | host-side loop | $O(G)$ device arrays |
| Padding waste | none | none |
| Load balance | OS scheduler per launch | static prefix or atomic queue |
| Algebra | $\prod_g C^{(g)}$ | identical |

Grouping changes **when and where** tiles run, never the bilinear map
$(A^{(g)},B^{(g)})\mapsto A^{(g)}B^{(g)}$.


In [ ]:
def naive_group_gemm_pseudo(problems):
    """
    Sequential torch.matmul for each (A, B) pair.

    problems: list of (A, B) with A:[Mi, Ki], B:[Ki, Ni]

    Each call is a separate cuBLAS/torch launch. Fine for a few large GEMMs;
    painful when you have hundreds of tiny ones (launch overhead dominates).
    """
    return [torch.matmul(a, b) for a, b in problems]


### Method 02 — Kernel-based (sequential Triton GEMM)

Each group is computed by the tutorial-03 Triton matmul, but the loop over $g$
still runs on the host and issues $G$ separate grids. This is the kernel-based
baseline: real Triton arithmetic, one launch per group.


In [ ]:
def group_gemm_sequential(problems):
    """
    Same loop, but each GEMM uses tutorial 03's Triton grouped matmul.
    (See markdown cells above for the math.)
    """
    return [matmul(a, b) for a, b in problems]


### Method 03 — Fused kernel-based (one launch)

All groups are packed into flat contiguous buffers. The host builds
$(M_g,N_g,K_g)$, base offsets, tile prefix sums, and a per-tile group id. One
grid of $T=\sum_gT_g$ programs walks the packed buffers and runs the familiar
$B_M\times B_N$ tile body. The launch count drops from $G$ to $1$; the algebra is
unchanged.


In [ ]:
@triton.jit
def group_gemm_packed_kernel(
    A_all,
    B_all,
    C_all,
    M_ptr,
    N_ptr,
    K_ptr,
    A_off_ptr,
    B_off_ptr,
    C_off_ptr,
    tile_off_ptr,
    group_ids,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """Method 03: one launch over all groups' tiles.

    Every group is packed into contiguous flat buffers.  The host supplies the
    per-group metadata, base offsets, tile prefix sums, and a per-tile group id.
    """
    pid = tl.program_id(0)
    g = tl.load(group_ids + pid)
    local = pid - tl.load(tile_off_ptr + g)

    M = tl.load(M_ptr + g)
    N = tl.load(N_ptr + g)
    K = tl.load(K_ptr + g)

    num_n = tl.cdiv(N, BLOCK_N)
    pid_m = local // num_n
    pid_n = local % num_n

    a_base = A_all + tl.load(A_off_ptr + g)
    b_base = B_all + tl.load(B_off_ptr + g)
    c_base = C_all + tl.load(C_off_ptr + g)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    a_ptrs = a_base + offs_m[:, None] * K + offs_k[None, :]
    b_ptrs = b_base + offs_k[:, None] * N + offs_n[None, :]
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k in range(0, tl.cdiv(K, BLOCK_K)):
        remaining = K - k * BLOCK_K
        a = tl.load(
            a_ptrs,
            mask=(offs_m[:, None] < M) & (offs_k[None, :] < remaining),
            other=0.0,
        )
        b = tl.load(
            b_ptrs,
            mask=(offs_k[:, None] < remaining) & (offs_n[None, :] < N),
            other=0.0,
        )
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K
        b_ptrs += BLOCK_K * N

    c = acc.to(C_all.dtype.element_ty)
    c_ptrs = c_base + offs_m[:, None] * N + offs_n[None, :]
    tl.store(
        c_ptrs,
        c,
        mask=(offs_m[:, None] < M) & (offs_n[None, :] < N),
    )


def group_gemm_one_launch(problems, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32):
    """Method 03 launcher: pack groups, build metadata, launch one grid."""
    if not problems:
        return []

    device = problems[0][0].device
    dtype = problems[0][0].dtype
    As = [a.contiguous() for a, _ in problems]
    Bs = [b.contiguous() for _, b in problems]
    Ms = [a.shape[0] for a in As]
    Ns = [b.shape[1] for b in Bs]
    Ks = [a.shape[1] for a in As]

    C_all = torch.empty(
        sum(m * n for m, n in zip(Ms, Ns)), device=device, dtype=dtype
    )
    Cs, c_off, off = [], [], 0
    for m, n in zip(Ms, Ns):
        Cs.append(C_all[off : off + m * n].view(m, n))
        c_off.append(off)
        off += m * n

    A_all = torch.cat([a.reshape(-1) for a in As])
    B_all = torch.cat([b.reshape(-1) for b in Bs])

    a_off, b_off, off = [], [], 0
    for a in As:
        a_off.append(off)
        off += a.numel()
    off = 0
    for b in Bs:
        b_off.append(off)
        off += b.numel()

    counts = [
        triton.cdiv(m, BLOCK_M) * triton.cdiv(n, BLOCK_N)
        for m, n in zip(Ms, Ns)
    ]
    tile_off = [0]
    for c in counts:
        tile_off.append(tile_off[-1] + c)
    total_tiles = tile_off[-1]

    def i32(values):
        return torch.tensor(values, device=device, dtype=torch.int32)

    group_ids = torch.repeat_interleave(
        torch.arange(len(problems), device=device),
        torch.tensor(counts, device=device, dtype=torch.int64),
    ).to(torch.int32)

    group_gemm_packed_kernel[(total_tiles,)](
        A_all,
        B_all,
        C_all,
        i32(Ms),
        i32(Ns),
        i32(Ks),
        i32(a_off),
        i32(b_off),
        i32(c_off),
        i32(tile_off),
        group_ids,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_K=BLOCK_K,
    )
    return Cs


### Pitfalls

- `torch.bmm` needs identical shapes — not a drop-in for MoE.
- "Grouped" in tutorial 03 ≠ Group GEMM in tutorial 08.
- Sequential launches hide true Group-GEMM benefits on many tiny problems.
- Forgetting the prefix-sum offset when mapping a global tile id.

### Self-check

1. Prove the block-diagonal identity (0.3).
2. List the metadata needed for one-launch Group GEMM.
3. Derive the prefix-sum tile map and prove it is a bijection.
4. When do $G$ host launches dominate runtime? Estimate the speedup.
5. Sketch a pointer table for 3 experts with different $M$.


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    problems = [
        (torch.randn(32, 16), torch.randn(16, 24)),
        (torch.randn(8, 32), torch.randn(32, 8)),
    ]
    outs = naive_group_gemm_pseudo(problems)
    for (a, b), c in zip(problems, outs):
        assert torch.allclose(c, a @ b, atol=1e-5)


def test_correctness() -> None:
    require_gpu()
    problems = [
        (
            torch.randn(128, 64, device=DEVICE, dtype=torch.float16),
            torch.randn(64, 256, device=DEVICE, dtype=torch.float16),
        ),
        (
            torch.randn(64, 128, device=DEVICE, dtype=torch.float16),
            torch.randn(128, 32, device=DEVICE, dtype=torch.float16),
        ),
    ]
    outs = group_gemm_sequential(problems)
    for (a, b), c in zip(problems, outs):
        ref = torch.matmul(a, b)
        assert torch.allclose(c.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("group gemm (sequential) OK", len(outs))


In [ ]:
def test_group_gemm_one_launch() -> None:
    """Method 03 (one launch) vs the per-group references."""
    require_gpu()
    torch.manual_seed(0)
    problems = [
        (
            torch.randn(128, 64, device=DEVICE, dtype=torch.float16),
            torch.randn(64, 256, device=DEVICE, dtype=torch.float16),
        ),
        (
            torch.randn(64, 128, device=DEVICE, dtype=torch.float16),
            torch.randn(128, 32, device=DEVICE, dtype=torch.float16),
        ),
    ]
    outs = group_gemm_one_launch(problems)
    for (a, b), c in zip(problems, outs):
        ref = torch.matmul(a, b)
        assert torch.allclose(c.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("group GEMM one-launch OK")
